In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import joblib
from scipy.sparse import save_npz, load_npz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [2]:
DATA_DIR = Path("../data")
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
resume_path = DATA_DIR / "processed" / "clean_resume.csv"
job_path = DATA_DIR / "processed" / "clean_job.csv"
if not resume_path.exists():
    resume_path = DATA_DIR / "clean_resume.csv"
if not job_path.exists():
    job_path = DATA_DIR / "clean_job.csv"
resume_df = pd.read_csv(resume_path)
job_df = pd.read_csv(job_path)

In [4]:
print(resume_df.shape)
print(job_df.shape)

(2481, 5)
(180370, 15)


In [5]:
required_resume_cols = ["resume_id", "category", "clean_resume"]
required_job_cols = ["job_id", "title", "full_job"]
for col in required_resume_cols:
    if col not in resume_df.columns:
        raise ValueError(f"Missing required resume column: {col}")
for col in required_job_cols:
    if col not in job_df.columns:
        raise ValueError(f"Missing required job column: {col}")
resume_df["clean_resume"] = resume_df["clean_resume"].fillna("").astype(str)
job_df["full_job"] = job_df["full_job"].fillna("").astype(str)
job_df["title"] = job_df["title"].fillna("unknown").astype(str)
resume_df.head(2), job_df[["job_id", "title", "full_job"]].head(2)

(   resume_id category                                             resume  \
 0   16852973       HR           HR ADMINISTRATOR/MARKETING ASSOCIATE\...   
 1   22323967       HR           HR SPECIALIST, US HR OPERATIONS      ...   
 
                                         clean_resume  resume_word_count  
 0  hr administrator marketing associate hr admini...                658  
 1  hr specialist us hr operations summary versati...                707  ,
    job_id                         title  \
 0       0  Digital Marketing Specialist   
 1       1                 Web Developer   
 
                                             full_job  
 0  digital marketing specialist digital marketing...  
 1  web developer web developer frontend web devel...  )

In [6]:
def normalize_tech_terms(text: str) -> str:
    text = str(text)
    replacements = {
        "c++": " cpp ",
        "c#": " csharp ",
        ".net": " dotnet ",
        "asp.net": " aspdotnet ",
        "node.js": " nodejs ",
        "react.js": " reactjs ",
        "next.js": " nextjs ",
        "vue.js": " vuejs ",
        "express.js": " expressjs ",
        "machine-learning": " machine learning ",
        "deep-learning": " deep learning ",
    }
    text = text.lower()
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text
def clean_text_light(text: str) -> str:
    text = normalize_tech_terms(text)

    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\b\d{10,}\b", " ", text)
    text = re.sub(r"[^a-z0-9+#\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [7]:
resume_texts = resume_df["clean_resume"].tolist()
job_texts = job_df["full_job"].tolist()

In [8]:
print("Num resumes:", len(resume_texts))
print("Num jobs:", len(job_texts))

Num resumes: 2481
Num jobs: 180370


In [9]:
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    stop_words="english",
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    norm="l2",
    dtype=np.float32,
    token_pattern=r"(?u)\b\w+\b",
)

job_vectors = vectorizer.fit_transform(job_texts)
resume_vectors = vectorizer.transform(resume_texts)

In [10]:
print("Job vector shape:", job_vectors.shape)
print("Resume vector shape:", resume_vectors.shape)

Job vector shape: (180370, 18916)
Resume vector shape: (2481, 18916)


In [11]:
joblib.dump(vectorizer, MODEL_DIR / "tfidf_vectorizer.pkl")
save_npz(MODEL_DIR / "tfidf_job_vectors.npz", job_vectors)
save_npz(MODEL_DIR / "tfidf_resume_vectors.npz", resume_vectors)

job_df[["job_id", "title", "full_job"]].to_csv(MODEL_DIR / "tfidf_job_index.csv", index=False)
resume_df[["resume_id", "category", "clean_resume"]].to_csv(MODEL_DIR / "tfidf_resume_index.csv", index=False)

print("Artifacts saved in:", MODEL_DIR)

Artifacts saved in: ..\models


In [12]:
def safe_top_k_indices(scores, top_k=5):
    scores = np.asarray(scores)

    if scores.size == 0:
        return np.array([], dtype=int)

    top_k = min(top_k, scores.size)

    idx = np.argpartition(scores, -top_k)[-top_k:]
    idx = idx[np.argsort(scores[idx])[::-1]]
    return idx


def lexical_score_to_percent(score: float) -> float:
    score = max(0.0, min(1.0, float(score)))
    return round(score * 100, 2)


def _safe_job_id(value):
    try:
        return int(value)
    except Exception:
        return value


def _safe_resume_id(value):
    try:
        return int(value)
    except Exception:
        return value

In [13]:
def get_top_jobs(resume_text, top_k=5, unique_titles=True):
    if not isinstance(resume_text, str) or not resume_text.strip():
        return pd.DataFrame(columns=["job_id", "job_title", "baseline_tfidf_score"])

    clean_resume = clean_text_light(resume_text)
    query_vec = vectorizer.transform([clean_resume])

    # TF-IDF vectors are l2-normalized, so linear_kernel = cosine similarity
    sim_scores = linear_kernel(query_vec, job_vectors).ravel()

    raw_k = min(len(sim_scores), max(top_k * 5, top_k))
    candidate_idx = safe_top_k_indices(sim_scores, raw_k)

    results = []
    seen_titles = set()

    for idx in candidate_idx:
        row = job_df.iloc[idx]
        title_key = str(row["title"]).strip().lower()

        if unique_titles and title_key in seen_titles:
            continue
        seen_titles.add(title_key)

        results.append({
            "job_id": _safe_job_id(row["job_id"]),
            "job_title": row["title"],
            "baseline_tfidf_score": lexical_score_to_percent(sim_scores[idx]),
        })

        if len(results) == top_k:
            break

    return pd.DataFrame(results)

In [14]:
def get_top_resumes(job_text, top_k=5):
    if not isinstance(job_text, str) or not job_text.strip():
        return pd.DataFrame(columns=["resume_id", "category", "baseline_tfidf_score", "resume_preview"])

    clean_job = clean_text_light(job_text)
    query_vec = vectorizer.transform([clean_job])

    sim_scores = linear_kernel(query_vec, resume_vectors).ravel()

    raw_k = min(len(sim_scores), max(top_k * 5, top_k))
    candidate_idx = safe_top_k_indices(sim_scores, raw_k)

    results = []

    for idx in candidate_idx:
        row = resume_df.iloc[idx]

        preview = row["clean_resume"][:180]
        if len(row["clean_resume"]) > 180:
            preview += "..."

        results.append({
            "resume_id": _safe_resume_id(row["resume_id"]),
            "category": row["category"],
            "baseline_tfidf_score": lexical_score_to_percent(sim_scores[idx]),
            "resume_preview": preview,
        })

        if len(results) == top_k:
            break

    return pd.DataFrame(results)

In [15]:
def match_resume_to_job_text(resume_text, job_text):
    if not isinstance(resume_text, str) or not resume_text.strip():
        raise ValueError("resume_text cannot be empty")

    if not isinstance(job_text, str) or not job_text.strip():
        raise ValueError("job_text cannot be empty")

    clean_resume = clean_text_light(resume_text)
    clean_job = clean_text_light(job_text)

    resume_vec = vectorizer.transform([clean_resume])
    job_vec = vectorizer.transform([clean_job])

    score = linear_kernel(resume_vec, job_vec)[0][0]

    return {
        "baseline_tfidf_score": lexical_score_to_percent(score)
    }

In [16]:
def top_jobs_for_existing_resume(resume_id, top_k=5, unique_titles=True):
    match = resume_df[resume_df["resume_id"] == resume_id]

    if match.empty:
        raise ValueError(f"resume_id not found: {resume_id}")

    resume_text = match.iloc[0]["clean_resume"]
    return get_top_jobs(resume_text, top_k=top_k, unique_titles=unique_titles)


def top_resumes_for_existing_job(job_id, top_k=5):
    match = job_df[job_df["job_id"] == job_id]

    if match.empty:
        raise ValueError(f"job_id not found: {job_id}")

    job_text = match.iloc[0]["full_job"]
    return get_top_resumes(job_text, top_k=top_k)

In [17]:
print("Top jobs for first resume:")
display(get_top_jobs(resume_df.iloc[0]["clean_resume"], top_k=5))

print("Top resumes for first job:")
display(get_top_resumes(job_df.iloc[0]["full_job"], top_k=5))

print("Direct pair score example:")
print(match_resume_to_job_text(
    resume_df.iloc[0]["clean_resume"],
    job_df.iloc[0]["full_job"]
))

Top jobs for first resume:


,job_id,job_title,baseline_tfidf_score
0,154372,Human Resources Manager,17.43


Top resumes for first job:


,resume_id,category,baseline_tfidf_score,resume_preview
0,15479281,APPAREL,28.25,pr event manager summary experienced creative ...
1,22380187,DIGITAL-MEDIA,25.37,social media coordinator highlights microsoft ...
2,13837784,DIGITAL-MEDIA,25.12,digital media buyer professional summary versa...
3,59011090,DIGITAL-MEDIA,23.45,social media coordinator summary to secure a p...
4,16536141,DIGITAL-MEDIA,23.41,interim senior digital marketing strategy mana...


Direct pair score example:
{'baseline_tfidf_score': 3.87}


In [18]:
def load_phase2_artifacts(model_dir=MODEL_DIR):
    vectorizer = joblib.load(model_dir / "tfidf_vectorizer.pkl")
    job_vectors = load_npz(model_dir / "tfidf_job_vectors.npz")
    resume_vectors = load_npz(model_dir / "tfidf_resume_vectors.npz")
    job_index = pd.read_csv(model_dir / "tfidf_job_index.csv")
    resume_index = pd.read_csv(model_dir / "tfidf_resume_index.csv")

    return vectorizer, job_vectors, resume_vectors, job_index, resume_index